# Model Evaluation / Backtesting

Mohd Yah-Ya Raiyan

**Purpose:** walk-forward backtesting and full metric computation (MAE,
RMSE, MAPE, R², directional accuracy) for every model family (Section 3.9,
and the web app's Model Accuracy page).

In [1]:

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

def compute_metrics(actual, pred):
    actual = np.array(actual, dtype=float); pred = np.array(pred, dtype=float)
    mask = ~(np.isnan(actual) | np.isnan(pred))
    actual, pred = actual[mask], pred[mask]
    resid = actual - pred
    rmse = float(np.sqrt(np.mean(resid**2)))
    mae = float(np.mean(np.abs(resid)))
    denom = np.where(np.abs(actual) < 1e-6, np.nan, actual)
    mape = float(np.nanmean(np.abs(resid/denom)) * 100)
    ss_res = np.sum(resid**2); ss_tot = np.sum((actual - actual.mean())**2)
    r2 = float(1 - ss_res/ss_tot)
    dir_acc = float(np.mean(np.sign(actual) == np.sign(pred)) * 100)
    return dict(RMSE=round(rmse,2), MAE=round(mae,2), MAPE=round(mape,1), R2=round(r2,3), DirAcc=round(dir_acc,1), N=int(len(actual)))


## Baseline and statistical models

Computed from real walk-forward backtest predictions (830 suburbs for Naive/Seasonal-naive/SARIMA, 149 suburbs for Prophet).

In [2]:

sarima_df = pd.read_csv("data/backtest_SARIMA_830suburbs.csv")
prophet_df = pd.read_csv("data/backtest_Prophet_149suburbs.csv")

results = {
    "Naive persistence": compute_metrics(sarima_df["actual"], sarima_df["naive"]),
    "Seasonal-naive": compute_metrics(sarima_df["actual"], sarima_df["seasonal_naive"]),
    "SARIMA": compute_metrics(sarima_df["actual"], sarima_df["sarima"]),
    "Prophet": compute_metrics(prophet_df["actual"], prophet_df["prophet"]),
}
pd.DataFrame(results).T


,RMSE,MAE,MAPE,R2,DirAcc,N
Naive persistence,27.09,17.16,1738.0,-1.224,47.6,14372.0
Seasonal-naive,25.26,16.42,3611.6,-0.934,50.4,14372.0
SARIMA,21.28,12.23,503.4,-0.372,50.8,14372.0
Prophet,93.49,15.58,3129.0,-0.012,51.7,1984.0


## Gradient Boosting — real walk-forward backtest

Rolling-origin backtest: for each of several cutoff quarters, train on
everything before the cutoff, predict the next quarter for every liquid
suburb, record actual vs. predicted, then roll the cutoff forward one
quarter and repeat. This is the same evaluation protocol as the SARIMA/
Prophet backtests above, applied to Gradient Boosting.

In [3]:

from sklearn.ensemble import GradientBoostingRegressor

fs = pd.read_csv("data/NSW_feature_store.csv", parse_dates=["quarter"])

feature_cols = ["lag_1_qoq_pct","lag_2_qoq_pct","lag_3_qoq_pct","lag_4_qoq_pct",
                "rolling_mean_4q","rolling_std_4q","state_benchmark_growth",
                "n_sales","flag_covid_period","flag_rate_hike_period",
                "is_q1","is_q2","is_q3","is_q4"]
target_col = "qoq_pct_change"

model_df = fs.dropna(subset=feature_cols + [target_col]).copy()
liquid = model_df.groupby("suburb")["n_sales"].mean()
liquid_suburbs = liquid[liquid >= 20].index
model_df = model_df[model_df["suburb"].isin(liquid_suburbs)].sort_values("quarter")

# Walk-forward: roll through the last 8 available quarters as test cutoffs
test_quarters = sorted(model_df["quarter"].unique())[-8:]
print(f"Walk-forward test quarters: {[q.strftime('%Y-%m') for q in test_quarters]}")
print(f"Liquid suburbs: {len(liquid_suburbs):,}")

all_actual, all_pred, all_suburb, all_quarter = [], [], [], []

for cutoff in test_quarters:
    train_fold = model_df[model_df["quarter"] < cutoff]
    test_fold = model_df[model_df["quarter"] == cutoff]
    if len(train_fold) < 500 or len(test_fold) == 0:
        continue
    gb_fold = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
    gb_fold.fit(train_fold[feature_cols], train_fold[target_col])
    preds = gb_fold.predict(test_fold[feature_cols])

    all_actual.extend(test_fold[target_col].tolist())
    all_pred.extend(preds.tolist())
    all_suburb.extend(test_fold["suburb"].tolist())
    all_quarter.extend(test_fold["quarter"].tolist())

gb_raw = pd.DataFrame({"suburb": all_suburb, "quarter": all_quarter, "actual": all_actual, "gradient_boosting": all_pred})
print(f"\nTotal walk-forward predictions collected: {len(gb_raw):,}")
gb_raw.head()


Walk-forward test quarters: ['2024-04', '2024-07', '2024-10', '2025-01', '2025-04', '2025-07', '2025-10', '2026-01']
Liquid suburbs: 575



Total walk-forward predictions collected: 4,629


,suburb,quarter,actual,gradient_boosting
0,JANNALI,2024-04-01,-4.934211,-2.229744
1,WEST TAMWORTH,2024-04-01,14.828897,8.430804
2,WESTMEAD,2024-04-01,5.860806,4.697082
3,SURRY HILLS,2024-04-01,1.219512,-0.218379
4,JAMISONTOWN,2024-04-01,-16.478555,11.986312


In [4]:

gb_metrics = compute_metrics(gb_raw["actual"], gb_raw["gradient_boosting"])
results["Gradient Boosting"] = gb_metrics
print("Gradient Boosting — real walk-forward backtest results:")
gb_metrics


Gradient Boosting — real walk-forward backtest results:


{'RMSE': 14.22,
 'MAE': 9.02,
 'MAPE': 171.0,
 'R2': 0.342,
 'DirAcc': 66.4,
 'N': 4629}

This fills in what was previously marked "pending" in the web app — real,
computed R² and directional accuracy for Gradient Boosting, not aggregate-only
metrics. The raw predictions are exported below so `webapp/app.py` can be
updated to show these instead of "pending".

In [5]:

gb_raw.to_csv("backtest_GB_raw_predictions.csv", index=False)
print(f"Wrote {len(gb_raw):,} rows to backtest_GB_raw_predictions.csv")


Wrote 4,629 rows to backtest_GB_raw_predictions.csv


## Full comparison table

In [6]:

final_table = pd.DataFrame(results).T
final_table = final_table.sort_values("RMSE")
final_table


,RMSE,MAE,MAPE,R2,DirAcc,N
Gradient Boosting,14.22,9.02,171.0,0.342,66.4,4629.0
SARIMA,21.28,12.23,503.4,-0.372,50.8,14372.0
Seasonal-naive,25.26,16.42,3611.6,-0.934,50.4,14372.0
Naive persistence,27.09,17.16,1738.0,-1.224,47.6,14372.0
Prophet,93.49,15.58,3129.0,-0.012,51.7,1984.0


## Interpretation

Gradient Boosting has the lowest RMSE and MAE of all models tested, and —
now that we can measure it — a **positive R²** (unlike the negative R² for
the statistical baselines) and directional accuracy meaningfully above 50%
on this walk-forward evaluation. This is stronger evidence for the model
choice than the RMSE/MAE-only comparison in the original report, and
should be used to update Section 3.9.3 and the web app's Model Accuracy
page once reviewed by the team.

Note: these walk-forward numbers may differ slightly from the single-split
demo in `forecasting.ipynb` since a walk-forward backtest averages across
several rolling cutoffs rather than one fixed train/test boundary — this
is the more rigorous and correct evaluation of the two.